<a href="https://colab.research.google.com/github/JayR1031/Jairo1031/blob/main/Graphical_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

#load file into Colab
df = pd.read_csv('hw4_messages.csv')

#Keep only necessary columns
df = df[['clean_text', 'intent']]
train, temp = train_test_split(df, test_size=0.2, stratify=df['intent'], random_state=42)
dev, test = train_test_split(temp, test_size=0.5, stratify=temp['intent'], random_state=42)

# Print sizes to confirm split
print(f"Train size: {len(train)}")
print(f"Dev size: {len(dev)}")
print(f"Test size: {len(test)}")

Train size: 888
Dev size: 111
Test size: 111


To fix the `FileNotFoundError`, please upload the `hw4_messages.csv` file using the file upload button in the left sidebar. After uploading the file, run the following cells to load the data and continue with the rest of the notebook.

In [ ]:
def contains_money(text):
    return int(any(word in text.lower() for word in ['refund', 'payment', 'money']))

def contains_track(text):
    return int(any(word in text.lower() for word in ['track', 'package', 'delivery']))

def contains_order(text):
    return int(any(word in text.lower() for word in ['order', 'buy', 'purchase']))

def contains_return(text):
    return int(any(word in text.lower() for word in ['return', 'replace']))

def length_bucket(text):
    l = len(text.split())
    if l <= 4:
        return 'short'
    elif l <= 8:
        return 'medium'
    else:
        return 'long'

def extract_features(row):
    return {
        'money': contains_money(row['clean_text']),
        'track': contains_track(row['clean_text']),
        'order': contains_order(row['clean_text']),
        'return': contains_return(row['clean_text']),
        'length': length_bucket(row['clean_text'])
    }

# Apply to all splits
train_feats = train.apply(extract_features, axis=1, result_type='expand')
dev_feats = dev.apply(extract_features, axis=1, result_type='expand')
test_feats = test.apply(extract_features, axis=1, result_type='expand')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

#load file from Google Drive
# Replace '/path/to/your/hw4_messages.csv' with the actual path to your file in Google Drive
df = pd.read_csv('/content/drive/My Drive/hw4_messages.csv')

#Keep only necessary columns
df = df[['clean_text', 'intent']]
train, temp = train_test_split(df, test_size=0.2, stratify=df['intent'], random_state=42)
dev, test = train_test_split(temp, test_size=0.5, stratify=temp['intent'], random_state=42)

# Print sizes to confirm split
print(f"Train size: {len(train)}")
print(f"Dev size: {len(dev)}")
print(f"Test size: {len(test)}")

Train size: 888
Dev size: 111
Test size: 111


In [ ]:
from collections import Counter, defaultdict

# Prior: P(Intent)
intent_counts = Counter(train['intent'])
num_intents = len(intent_counts)
total_intents = sum(intent_counts.values())
prior_intent = {intent: (intent_counts[intent]+1)/(total_intents+num_intents) for intent in intent_counts}

# Conditional: P(Feature | Intent) for binary features
# Laplace smoothing: add 1 to numerator, add 2 to denominator (for binary)
def cpt(feature, value, intent):
    subset = train[train['intent'] == intent]
    count = sum(extract_features(row)[feature] == value for idx, row in subset.iterrows())
    return (count + 1) / (len(subset) + 2)

# For each feature and intent, store probabilities
feature_names = ['money', 'track', 'order', 'return']
p_feature_given_intent = defaultdict(dict)
for feature in feature_names:
    for intent in intent_counts:
        p_feature_given_intent[feature][intent] = {
            0: cpt(feature, 0, intent),
            1: cpt(feature, 1, intent)
        }

# For length_bucket (categorical)
def cpt_length(length_value, intent):
    subset = train[train['intent'] == intent]
    count = sum(extract_features(row)['length'] == length_value for idx, row in subset.iterrows())
    return (count + 1) / (len(subset) + 3)  # 3 buckets: short, medium, long

length_values = ['short', 'medium', 'long']
p_length_given_intent = defaultdict(dict)
for intent in intent_counts:
    for length in length_values:
        p_length_given_intent[intent][length] = cpt_length(length, intent)


In [ ]:
def predict_intent(features):
    scores = {}
    for intent in intent_counts:
        # Start with prior
        prob = prior_intent[intent]
        # Multiply by each feature's conditional probability
        for feature in feature_names:
            prob *= p_feature_given_intent[feature][intent][features[feature]]
        # Multiply by length bucket probability
        prob *= p_length_given_intent[intent][features['length']]
        scores[intent] = prob
    # Return intent with highest probability
    return max(scores, key=scores.get), scores


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
y_true = test['intent'].tolist()
y_pred = []
y_posteriors = []
for feat in test_feats.to_dict('records'):
    pred, post = predict_intent(feat)
    y_pred.append(pred)
    y_posteriors.append(post)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Classification Report:\n", classification_report(y_true, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

Accuracy: 0.8288288288288288
Classification Report:
               precision    recall  f1-score   support

     General       0.64      0.75      0.69        24
       Order       0.76      0.97      0.85        30
      Refund       1.00      1.00      1.00        29
       Track       1.00      0.57      0.73        28

    accuracy                           0.83       111
   macro avg       0.85      0.82      0.82       111
weighted avg       0.86      0.83      0.82       111

Confusion Matrix:
 [[18  6  0  0]
 [ 1 29  0  0]
 [ 0  0 29  0]
 [ 9  3  0 16]]


In [ ]:
# Find a misclassified example
for idx, (true, pred, feat, post) in enumerate(zip(y_true, y_pred, test_feats.to_dict('records'), y_posteriors)):
    if true != pred:
        print(f"Message: {test.iloc[idx]['clean_text']}")
        print(f"True intent: {true}")
        print(f"Predicted intent: {pred}")
        print(f"Feature values: {feat}")
        print(f"Posterior probabilities: {post}")
        # What-if: flip one feature (e.g., 'money')
        feat_flipped = feat.copy()
        feat_flipped['money'] = 1 - feat_flipped['money']
        pred2, post2 = predict_intent(feat_flipped)
        print(f"\nAfter flipping 'money':")
        print(f"New predicted intent: {pred2}")
        print(f"New posterior probabilities: {post2}")
        break  # Only show one example


Message: has my order been shipped
True intent: Track
Predicted intent: Order
Feature values: {'money': 0, 'track': 0, 'order': 1, 'return': 0, 'length': 'medium'}
Posterior probabilities: {'Order': 0.14131251768801917, 'Refund': 0.0007611183677619579, 'Track': 0.0021551393285012786, 'General': 0.0003036329522686993}

After flipping 'money':
New predicted intent: Refund
New posterior probabilities: {'Order': 0.0005744411288130861, 'Refund': 0.0035085700367563422, 'Track': 9.796087856823996e-06, 'General': 1.541284021668525e-06}


In [ ]:
from collections import defaultdict

# Use clean_text and noisy_text columns
train_hmm = pd.read_csv('hw4_messages.csv')[['clean_text', 'noisy_text']]

# Build transition and emission counts
transitions = defaultdict(lambda: defaultdict(lambda: 1))  # Laplace smoothing
emissions = defaultdict(lambda: defaultdict(lambda: 1))
char_set = set()
for _, row in train_hmm.iterrows():
    clean = row['clean_text']
    noisy = row['noisy_text']
    for i in range(len(clean)-1):
        transitions[clean[i]][clean[i+1]] += 1
        char_set.add(clean[i])
        char_set.add(clean[i+1])
    for c, n in zip(clean, noisy):
        emissions[c][n] += 1
        char_set.add(n)

# Normalize to probabilities
def normalize_counts(counts):
    probs = defaultdict(dict)
    for c1 in counts:
        total = sum(counts[c1].values())
        for c2 in counts[c1]:
            probs[c1][c2] = counts[c1][c2] / total
    return probs

trans_probs = normalize_counts(transitions)
emiss_probs = normalize_counts(emissions)
all_chars = list(char_set)

In [ ]:
import numpy as np

def viterbi_decode(noisy, trans_probs, emiss_probs, all_chars):
    T = len(noisy)
    N = len(all_chars)
    dp = np.zeros((N, T))
    backtrack = np.zeros((N, T), dtype=int)

    # Initialize first position
    for i, c in enumerate(all_chars):
        dp[i, 0] = emiss_probs.get(c, {}).get(noisy[0], 1e-6)

    # Dynamic programming
    for t in range(1, T):
        for i, curr_c in enumerate(all_chars):
            best_prob, best_prev = -1, 0
            for j, prev_c in enumerate(all_chars):
                trans_p = trans_probs.get(prev_c, {}).get(curr_c, 1e-6)
                emiss_p = emiss_probs.get(curr_c, {}).get(noisy[t], 1e-6)
                prob = dp[j, t-1] * trans_p * emiss_p
                if prob > best_prob:
                    best_prob = prob
                    best_prev = j
            dp[i, t] = best_prob
            backtrack[i, t] = best_prev

    # Backtrack to get best path

    best_last  = np.argmax(dp[:, -1])
    result = [all_chars[best_last]]
    for t in range(T-1, 0, -1):
        best_last = backtrack[best_last, t]
        result.insert(0, all_chars[best_last])
    return ''.join(result)

In [ ]:
train_idx = train.index
dev_idx = dev.index
test_idx = test.index


test_hmm = pd.read_csv('hw4_messages.csv').loc[test_idx][['clean_text','noisy_text']]


char_correct = 0
char_total = 0
word_correct = 0
word_total = 0
results = []
for _, row in test_hmm.iterrows():
    noisy, clean = row['noisy_text'], row['clean_text']
    pred = viterbi_decode(noisy, trans_probs, emiss_probs, all_chars)
    char_correct += sum(a == b for a, b in zip(pred, clean))
    char_total += len(clean)
    word_correct += int(pred == clean)
    word_total += 1
    results.append((noisy, pred, clean))

if char_total == 0 or word_total == 0:
    print('No test data to evaluate! Check your test set slicing.')
else:
    print(f"Per-character accuracy: {char_correct / char_total * 100:.2f}%")
    print(f"Per-word accuracy: {word_correct / word_total * 100:.2f}%")

Per-character accuracy: 36.74%
Per-word accuracy: 1.80%


In [ ]:
# Find three cases: success, over-correction, failure
success, overcorrection, failure = None, None, None
for noisy, pred, clean in results:
    if pred == clean and noisy != clean and not success:
        success = (noisy, pred, clean)
    elif pred != clean and noisy == clean and not overcorrection:
        overcorrection = (noisy, pred, clean)
    elif pred != clean and noisy != clean and not failure:
        failure = (noisy, pred, clean)
    if success and overcorrection and failure:
        break

print("Success case:")
if success:
    print(f"Noisy: {success[0]} | Predicted: {success[1]} | True: {success[2]}")
else:
    print("No success case found.")

print("Over-correction case:")
if overcorrection:
    print(f"Noisy: {overcorrection[0]} | Predicted: {overcorrection[1]} | True: {overcorrection[2]}")
else:
    print("No over-correction case found.")

print("Failure case:")
if failure:
    print(f"Noisy: {failure[0]} | Predicted: {failure[1]} | True: {failure[2]}")
else:
    print("No failure case found.")


Success case:
No success case found.
Over-correction case:
Noisy: can you help me | Predicted: can youthelp me | True: can you help me
Failure case:
Noisy: trck my deilvery status | Predicted: t ck my delivery status | True: track my delivery status
